In [ ]:
import pandas as pd
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

import matplotlib.pyplot as plt

In [ ]:
X_test = pd.read_csv("X_test_processed.csv")

y_test = pd.read_csv("y_test.csv").squeeze()

print("Test Dataset Loaded Successfully")

In [ ]:
logistic = joblib.load("logistic_regression.pkl")

decision = joblib.load("decision_tree.pkl")

randomforest = joblib.load("random_forest.pkl")

knn = joblib.load("knn.pkl")

svm = joblib.load("svm.pkl")

naive = joblib.load("naive_bayes.pkl")

print("All Models Loaded Successfully")

In [ ]:
models = {

"Logistic Regression": logistic,

"Decision Tree": decision,

"Random Forest": randomforest,

"KNN": knn,

"SVM": svm,

"Naive Bayes": naive

}

In [ ]:
results = []

best_accuracy = 0

best_model = None

best_model_name = ""

for name, model in models.items():

    print("="*50)

    print(name)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    precision = precision_score(
        y_test,
        predictions,
        pos_label="ckd"
    )

    recall = recall_score(
        y_test,
        predictions,
        pos_label="ckd"
    )

    f1 = f1_score(
        y_test,
        predictions,
        pos_label="ckd"
    )

    print("Accuracy :", accuracy)

    print("Precision :", precision)

    print("Recall :", recall)

    print("F1 Score :", f1)

    print()

    print("Confusion Matrix")

    print(confusion_matrix(y_test, predictions))

    print()

    print(classification_report(y_test, predictions))

    # ROC-AUC only for models supporting predict_proba
    if hasattr(model, "predict_proba"):
        probability = model.predict_proba(X_test)[:,1]

        auc = roc_auc_score(y_test, probability)

    else:
        auc = None

    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1,
        auc
    ])

    if accuracy > best_accuracy:

        best_accuracy = accuracy

        best_model = model

        best_model_name = name

In [ ]:
comparison = pd.DataFrame(

results,

columns=[

"Model",

"Accuracy",

"Precision",

"Recall",

"F1 Score",

"AUC"

]

)

print(comparison)

In [ ]:
print()

print("Best Model")

print(best_model_name)

print("Accuracy =", best_accuracy)

In [ ]:
joblib.dump(

best_model,

"best_model.pkl"

)

print("Best Model Saved Successfully")

In [ ]:
plt.figure(figsize=(8,6))

for name, model in models.items():

    if hasattr(model, "predict_proba"):

        probability = model.predict_proba(X_test)[:,1]

        fpr, tpr, _ = roc_curve(

            y_test,

            probability,

            pos_label="ckd"

        )

        plt.plot(

            fpr,

            tpr,

            label=name

        )

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve Comparison")

plt.legend()

plt.grid()

plt.show()